In [322]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
import unicodedata
import numpy as np
import os
import io
import time
import regex as re
from indicnlp.tokenize.indic_tokenize import trivial_tokenize
from indicnlp.tokenize.sentence_tokenize import sentence_split
from indicnlp.transliterate.unicode_transliterate import UnicodeIndicTransliterator

In [3]:
translation_dir = os.path.abspath("../../../translation/indic_languages_corpus/")
print(translation_dir)

/Users/deven/Developer/ML_AI/translation/indic_languages_corpus


In [5]:
for file in os.listdir(translation_dir):
    print(file)

monolingual
.DS_Store
bilingual


In [6]:
hindi_english = os.path.join(translation_dir, "bilingual/hi-en")

In [7]:
os.path.exists(hindi_english)

True

# **Data Preperation**

Once we have loaded the dataset, we preprocess the data as follows:

Add a start and end token to each sentence.

Clean the sentences by removing special characters.

Create a word index and reverse word index (dictionaries mapping from word → id and id → word).

Pad each sentence to a maximum length.

In [306]:
# Restrict the total number of sentences to 70000
train = []
test = []
with open(os.path.join(hindi_english, "train.en"), "r") as f:
    train = f.readlines()
    # x = train.split(".")
with open(os.path.join(hindi_english, "train.hi"), "r") as f:
    test = f.readlines()

input_sentences = []
output_sentences = []
NUM_SENTENCES = 70000
for line in train[:NUM_SENTENCES]:
    input_sentences.append(line.rstrip().strip("\n").strip("-"))
output_sentences = []
for line in test[:NUM_SENTENCES]:
    output_sentences.append(
        ["sos"] + trivial_tokenize(line.rstrip().strip("\n").strip("-")) + ["sos"]
    )

In [307]:
print(input_sentences[10])
print(output_sentences[10])

Miporol, extremely potent, will keep you functioning normally until your death.
['sos', 'Miporol', 'बहुत', 'मददगार', 'होगा', '.', 'आपके', 'शरीर', 'मृत्यु', 'तक', 'ठीक', 'काम', 'करेगा', '.', 'sos']


In [308]:
print("num samples input:", len(input_sentences))
print("num samples output:", len(output_sentences))

num samples input: 70000
num samples output: 70000


In [309]:
print(input_sentences[-1])
print(output_sentences[-1])

Her face.
['sos', 'उसका', 'चेहरा', '.', 'sos']


In [310]:
print("num samples input:", len(input_sentences))
print("num samples output:", len(output_sentences))

num samples input: 70000
num samples output: 70000



| Abbr | Long | Description |
| ---- | --------------------- | -------------------------------------------------- |
| Lu | Uppercase_Letter | an uppercase letter |
| Ll | Lowercase_Letter | a lowercase letter |
| Lt | Titlecase_Letter | a digraphic character, with first part uppercase |
| LC | Cased_Letter | Lu \| Ll \| Lt |
| Lm | Modifier_Letter | a modifier letter |
| Lo | Other_Letter | other letters, including syllables and ideographs |
| L | Letter | Lu \| Ll \| Lt \| Lm \| Lo |
| Mn | Nonspacing_Mark | a nonspacing combining mark (zero advance width) |
| Mc | Spacing_Mark | a spacing combining mark (positive advance width) |
| Me | Enclosing_Mark | an enclosing combining mark |
| M | Mark | Mn \| Mc \| Me |
| Nd | Decimal_Number | a decimal digit |
| Nl | Letter_Number | a letterlike numeric character |
| No | Other_Number | a numeric character of other type |
| N | Number | Nd \| Nl \| No |
| Pc | Connector_Punctuation | a connecting punctuation mark, like a tie |
| Pd | Dash_Punctuation | a dash or hyphen punctuation mark |
| Ps | Open_Punctuation | an opening punctuation mark (of a pair) |
| Pe | Close_Punctuation | a closing punctuation mark (of a pair) |
| Pi | Initial_Punctuation | an initial quotation mark |
| Pf | Final_Punctuation | a final quotation mark |
| Po | Other_Punctuation | a punctuation mark of other type |
| P | Punctuation | Pc \| Pd \| Ps \| Pe \| Pi \| Pf \| Po |
| Sm | Math_Symbol | a symbol of mathematical use |
| Sc | Currency_Symbol | a currency sign |
| Sk | Modifier_Symbol | a non-letterlike modifier symbol |
| So | Other_Symbol | a symbol of other type |
| S | Symbol | Sm \| Sc \| Sk \| So |
| Zs | Space_Separator | a space character (of various non-zero widths) |
| Zl | Line_Separator | U+2028 LINE SEPARATOR only |
| Zp | Paragraph_Separator | U+2029 PARAGRAPH SEPARATOR only |
| Z | Separator | Zs \| Zl \| Zp |
| Cc | Control | a C0 or C1 control code |
| Cf | Format | a format control character |
| Cs | Surrogate | a surrogate code point |
| Co | Private_Use | a private-use character |
| Cn | Unassigned | a reserved unassigned code point or a noncharacter |
| C | Other | Cc \| Cf \| Cs \| Co \| Cn |



In [311]:
# Converts the unicode file to ascii
# Since the model is dealing with multilingual text so it will be important to standardize the input text.
# Unicode normalization splits accented characters and replace compatibility characters with their ASCII equivalents.
# https://bit.ly/2TnLffX
def unicode_to_ascii(s):
    return "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )


def preprocess_sentence(w):
    w = unicode_to_ascii(w.lower().strip())

    # creating a space between a word and the punctuation following it
    # eg: "he is a boy." => "he is a boy ."

    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    w = re.sub(r'[" "]+', " ", w)

    # replacing everything with space except (a-z, A-Z, ".", "?", "!", ",")
    w = re.sub(r"[^a-zA-Z?.!,¿]+", " ", w)

    w = w.strip()

    # adding a start and an end token to the sentence
    # so that the model know when to start and stop predicting.
    w = "<sos> " + w + " <eos>"
    return w

In [317]:
input_sentences = [preprocess_sentence(x) for x in input_sentences]

In [323]:
input_sentences[:5]

['<sos> sos and what is their sigil ? eos <eos>',
 '<sos> sos i do not want to die . eos <eos>',
 '<sos> sos it s the same country i think . eos <eos>',
 '<sos> sos then they ll be crying like babies . eos <eos>',
 '<sos> sos no , i need power up ! eos <eos>']

In [324]:
input_sentences[8]

'<sos> sos i told her we rest on sundays . eos <eos>'

In [325]:
print(input_sentences[-1])
print(output_sentences[-1])

<sos> sos her face . eos <eos>
['sos', 'उसका', 'चेहरा', '.', 'sos']


In [326]:
print(len(" ".join(output_sentences[0])))
x = unicode_to_ascii(" ".join(output_sentences[0]))  # Removes stop words
y = unicode_to_ascii("Hi hi hi?, ? > as 123 ఇద 0ి -+]")
print(x, "\n", y, "\n", len(x))

31
sos और उनक Sigil कया ह ? sos 
 Hi hi hi?, ? > as 123 ఇద 0 -+] 
 28


In [327]:
import string

sequence = [[1], [2, 3], [4, 5, 6]]
tf.keras.utils.pad_sequences(sequence)

array([[0, 0, 1],
       [0, 2, 3],
       [4, 5, 6]], dtype=int32)

In [328]:
def tokenize(lang, pad):
    lang_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters="")

    lang_tokenizer.fit_on_texts(lang)

    tensor = lang_tokenizer.texts_to_sequences(lang)

    tensor = tf.keras.preprocessing.sequence.pad_sequences(tensor, padding=pad)
    return tensor, lang_tokenizer

In [331]:
tensor, lang_tokenizer = tokenize("Hi how are you", pad="post")

In [332]:
tensor

array([[1],
       [3],
       [0],
       [1],
       [2],
       [4],
       [0],
       [5],
       [6],
       [7],
       [0],
       [8],
       [2],
       [9]], dtype=int32)

In [345]:
# def load_dataset(inp_lang,target_lang):
#     input_tensor,inp_lang_tensor = tokenize(inp_lang,'post')
#     output_tensor,output_lang_tensor = tokenize(target_lang,'post')
#     return input_tensor,inp_lang_tensor,output_tensor,output_lang_tensor
# function to call the tokenize function to perform tokenizing and padding


def load_dataset(inp_lang, targ_lang):
    # creating cleaned input, output pairs
    input_tensor, inp_lang_tokenizer = tokenize(inp_lang, "post")
    target_tensor, targ_lang_tokenizer = tokenize(targ_lang, "post")

    return input_tensor, target_tensor, inp_lang_tokenizer, targ_lang_tokenizer

In [346]:
input_tensor, target_tensor, inp_lang, targ_lang = load_dataset(
    input_sentences, output_sentences
)

# Calculate max_length of the target tensors
# For our project, the max_length_targ and max_length_inp are 69 and 72 respectively.

max_length_targ, max_length_inp = target_tensor.shape[1], input_tensor.shape[1]
print(max_length_targ)
print(max_length_inp)

69
74


In [338]:
max_input_shape, max_out_shape = input_tensor.shape[1], output_tensor.shape[1]
print(f"{max_input_shape},{max_out_shape}")

74,69


In [340]:
from IPython.display import display

display(input_tensor[9])
display(output_tensor[9])

array([   1,    2,    7,  108,   64,   65,  464, 6237,   23,    6,   61,
         10,    3,    4,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0], dtype=int32)

array([  1,  46, 202,  17, 202,  25,  38, 552,  78,  28,   4, 269,   7,
         1,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0], dtype=int32)

## Train Test Split

In [347]:
# Creating training and validation sets using an 80-20 split
input_tensor_train, input_tensor_val, target_tensor_train, target_tensor_val = (
    train_test_split(input_tensor, output_tensor, test_size=0.2, random_state=42)
)

# Show length
print(
    len(input_tensor_train),
    len(target_tensor_train),
    len(input_tensor_val),
    len(target_tensor_val),
)

56000 56000 14000 14000


In [348]:
# checking if the input sequences have been obtained and padded properly
x = input_tensor_val[9]
y = target_tensor_val[9]

In [349]:
# a function to test if the word to index / index to word mappings have been obtained correctly.
# representative output for two sample english and hindi sentences given in the code block below


def convert(lang, tensor):
    for t in tensor:
        if t != 0:
            print("%d ----> %s" % (t, lang.index_word[t]))
            print(
                "%s ----> %d"
                % (lang.index_word[t], lang.word_index[lang.index_word[t]])
            )

In [350]:
print("Input Language; index to word mapping")
convert(inp_lang, input_tensor_train[0])
print()
print("Target Language; index to word mapping")
convert(targ_lang, target_tensor_train[0])

Input Language; index to word mapping
1 ----> <sos>
<sos> ----> 1
2 ----> sos
sos ----> 2
47 ----> get
get ----> 47
58 ----> out
out ----> 58
19 ----> of
of ----> 19
17 ----> that
that ----> 17
1782 ----> van
van ----> 1782
15 ----> !
! ----> 15
3 ----> eos
eos ----> 3
4 ----> <eos>
<eos> ----> 4

Target Language; index to word mapping
1 ----> sos
sos ----> 1
48 ----> -
- ----> 48
21 ----> कि
कि ----> 21
2465 ----> वैन
वैन ----> 2465
17 ----> से
से ----> 17
79 ----> बाहर
बाहर ----> 79
1541 ----> निकलो
निकलो ----> 1541
13 ----> !
! ----> 13
1 ----> sos
sos ----> 1


In [357]:
print(list(zip(inp_lang.word_index.keys(), inp_lang.word_index.values()))[:5])
print(list(zip(targ_lang.word_index.keys(), targ_lang.word_index.values()))[:5])
print(len(inp_lang.word_index))
print(len(targ_lang.word_index))

[('<sos>', 1), ('sos', 2), ('eos', 3), ('<eos>', 4), ('.', 5)]
[('sos', 1), ('.', 2), ('है', 3), (',', 4), ('।', 5)]
17620
22222


In [364]:
# BUFFER_SIZE stores the number of training points
BUFFER_SIZE = len(input_tensor_train)
print(BUFFER_SIZE)
# BATCH_SIZE is set to 64. Training and gradient descent happens in batches of 64
BATCH_SIZE = 64

# the number of batches in one epoch (also, the number of steps during training, when we go batch by batch)
steps_per_epoch = BUFFER_SIZE // BATCH_SIZE

# the length of the embedded vector
embedding_dim = 256

# no of GRUs
units = 1024

# getting the size of the input and output vocabularies.
vocab_inp_size = len(inp_lang.word_index) + 1
vocab_tar_size = len(targ_lang.word_index) + 1

# now, we shuffle the dataset and split it into batches of 64
dataset = tf.data.Dataset.from_tensor_slices(
    (input_tensor_train, target_tensor_train)
).shuffle(BUFFER_SIZE)
dataset = dataset.batch(
    BATCH_SIZE, drop_remainder=True
)  # the remainder after splitting by 64 are dropped

print(BUFFER_SIZE)
print(BUFFER_SIZE // 64)
print(steps_per_epoch)

56000
56000
875
875


In [365]:
# to understand the shape of an input batch
example_input_batch, example_target_batch = next(iter(dataset))
example_input_batch.shape, example_target_batch.shape

(TensorShape([64, 74]), TensorShape([64, 69]))

<tf.Tensor: shape=(64, 74), dtype=int32, numpy=
array([[  1,   2,  56, ...,   0,   0,   0],
       [  1,   2,  95, ...,   0,   0,   0],
       [  1,   2,  14, ...,   0,   0,   0],
       ...,
       [  1,   2, 301, ...,   0,   0,   0],
       [  1,   2,  24, ...,   0,   0,   0],
       [  1,   2,  24, ...,   0,   0,   0]], dtype=int32)>

# Vectorization

In [121]:
# Vectorizing..
vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=100, standardize="lower_and_strip_punctuation", output_sequence_length=3
)


x1 = vectorize_layer.adapt(input_sentences[:5])

vectorize_layer.get_vocabulary()

In [148]:
english = []
for x in output_sentences:
    english.extend([" ".join(x)])
english_vectors = vectorize_layer(english)

<tf.Tensor: shape=(70000, 3), dtype=int64, numpy=
array([[ 2, 10, 20],
       [ 2,  4, 28],
       [ 2,  9,  7],
       ...,
       [ 2, 19,  7],
       [ 2,  4,  1],
       [ 2, 65,  1]])>

In [142]:
def textualize(tokenized_sentence):
    sentence = []
    for token in tokenized_sentence:
        sentence.append(vectorize_layer.get_vocabulary()[token])
    return " ".join(sentence)

In [152]:
for tokens in english_vectors[:30]:
    print(textualize(tokens.numpy()))

sos and what
sos i do
sos it s
sos then they
sos no i
sos i will
sos you [UNK]
sos no he
sos i [UNK]
sos you [UNK]
sos [UNK] [UNK]
sos i [UNK]
sos [UNK] [UNK]
sos your [UNK]
sos you can
sos you don
sos you [UNK]
sos i [UNK]
sos i [UNK]
sos tell me
sos you re
sos [UNK] do
sos what do
sos i know
sos don t
sos say [UNK]
sos it ll
sos it s
sos will i
sos did you


In [157]:
len(vectorize_layer.get_vocabulary())

100

In [160]:
z = set(vectorize_layer.get_vocabulary())

In [161]:
z

{'',
 '[UNK]',
 np.str_('a'),
 np.str_('about'),
 np.str_('all'),
 np.str_('an'),
 np.str_('and'),
 np.str_('are'),
 np.str_('as'),
 np.str_('at'),
 np.str_('back'),
 np.str_('be'),
 np.str_('been'),
 np.str_('but'),
 np.str_('can'),
 np.str_('come'),
 np.str_('did'),
 np.str_('do'),
 np.str_('don'),
 np.str_('down'),
 np.str_('for'),
 np.str_('from'),
 np.str_('get'),
 np.str_('go'),
 np.str_('going'),
 np.str_('gonna'),
 np.str_('good'),
 np.str_('got'),
 np.str_('have'),
 np.str_('he'),
 np.str_('her'),
 np.str_('here'),
 np.str_('him'),
 np.str_('his'),
 np.str_('how'),
 np.str_('i'),
 np.str_('if'),
 np.str_('in'),
 np.str_('is'),
 np.str_('it'),
 np.str_('just'),
 np.str_('know'),
 np.str_('let'),
 np.str_('like'),
 np.str_('ll'),
 np.str_('look'),
 np.str_('m'),
 np.str_('man'),
 np.str_('me'),
 np.str_('my'),
 np.str_('need'),
 np.str_('no'),
 np.str_('not'),
 np.str_('now'),
 np.str_('of'),
 np.str_('on'),
 np.str_('one'),
 np.str_('or'),
 np.str_('our'),
 np.str_('out'),
 np.

# Encoder Decoder Model

![Encoder Decoder Architecture](Encoder%20Decoder%20Architecture.png)

## Sample Model

In [378]:
from tensorflow import keras


class MySampleModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = tf.keras.layers.Dense(32, activation="relu")
        self.dense2 = tf.keras.layers.Dense(5, activation="softmax")
        self.dropout = tf.keras.layers.Dropout(0.5)

    def call(self, inputs, training=False):
        x = self.dense1(inputs)
        x = self.dropout(x, training=training)
        return self.dense2(x)

    def __str__(self):
        return "Sample My model"

In [379]:
model = MySampleModel()

## Encoder

In [394]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
        super(Encoder, self).__init__()
        self.batch_sz = batch_sz  # set batch size
        self.enc_units = enc_units  # set the number of GRU units
        self.embedding = tf.keras.layers.Embedding(
            vocab_size, embedding_dim
        )  # set the embedding layer using the input's vocabulary size and the embedding dimension (which is set to 256)
        self.gru = tf.keras.layers.GRU(
            self.enc_units,
            return_sequences=True,
            return_state=True,
            recurrent_initializer="glorot_uniform",
        )  # define the GRU layer

    def call(
        self, x, hidden
    ):  # this function is invoked when the function encoder is called with an input and an initialised hidden layer
        x = self.embedding(x)
        output, state = self.gru(
            x, initial_state=hidden
        )  # pass input x into the GRU layer
        return output, state  # function returns the encoder output and the hidden state

    def initialize_hidden_state(
        self,
    ):  # intialise hidden layer to all zeroes (for determining the shape)
        return tf.zeros((self.batch_sz, self.enc_units))

    def __str__(self):
        return "Upgrad Encoder"

### Gru usage

In [385]:
inputs = np.random.random((32, 10, 8))
gru = keras.layers.GRU(4, activation="tanh")
output = gru(inputs)
dir(output)

['OVERLOADABLE_OPERATORS',
 '_USE_EQUALITY',
 '__abs__',
 '__add__',
 '__and__',
 '__annotations__',
 '__array__',
 '__array_priority__',
 '__bool__',
 '__class__',
 '__complex__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__div__',
 '__dlpack__',
 '__dlpack_device__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__float__',
 '__floordiv__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__index__',
 '__init__',
 '__init_subclass__',
 '__int__',
 '__invert__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__matmul__',
 '__mod__',
 '__module__',
 '__mul__',
 '__ne__',
 '__neg__',
 '__new__',
 '__nonzero__',
 '__or__',
 '__pow__',
 '__radd__',
 '__rand__',
 '__rdiv__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rfloordiv__',
 '__rmatmul__',
 '__rmod__',
 '__rmul__',
 '__ror__',
 '__round__',
 '__rpow__',
 '__rsub__',
 '__rtruediv__',
 '__rxor__',
 '__setattr__',
 '__sizeof_

###  Encoder initializaion,

In [395]:
encoder = Encoder(
    vocab_inp_size, embedding_dim, units, BATCH_SIZE
)  # create an Encoder class object

In [396]:
sample_hidden = encoder.initialize_hidden_state()

In [397]:
sample_output, sample_hidden = encoder(example_input_batch, sample_hidden)
print(
    "Encoder output shape: (batch size, sequence length, units) {}".format(
        sample_output.shape
    )
)
print("Encoder Hidden state shape: (batch size, units) {}".format(sample_hidden.shape))

Encoder output shape: (batch size, sequence length, units) (64, 74, 1024)
Encoder Hidden state shape: (batch size, units) (64, 1024)


In [400]:
encoder.summary()

Model: "encoder_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (64, 74, 256)          │     4,510,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_3 (GRU)                     │ ((64, 74, 1024), (64,  │     3,938,304 │
│                                 │ 1024))                 │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,449,280 (32.23 MB)

 Trainable params: 8,449,280 (32.23 MB)

 Non-trainable params: 0 (0.00 B)

## Decoder`

In [401]:
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz):
        super(Decoder, self).__init__()
        self.batch_sz = batch_sz  # batch_size which is defined as 64
        self.dec_units = dec_units  # the number of decoder GRU units
        self.embedding = keras.layers.Embedding(
            vocab_size, embedding_dim
        )  # defining an embedding layer for the target language output.
        self.gru = keras.layers.GRU(
            self.dec_units,
            return_sequences=True,
            return_state=True,
            recurrent_initializer="glorot_uniform",
        )  # GRU layer
        self.fc = keras.layers.Dense(vocab_size)

    def call(self, x, hidden):

        # x shape after passing through embedding == (batch_size, 1, embedding_dim)
        x = self.embedding(x)  # creating an embedding layer for the target output

        # passing the initial state to the GRU as the hidden state
        output, state = self.gru(x, initial_state=hidden)

        # output shape == (batch_size * 1, hidden_size)
        output = tf.reshape(output, (-1, output.shape[2]))

        # output shape == (batch_size, vocab)
        x = self.fc(output)  # pass the output through the dense layer

        return x, state  # return decoder output and decoder state

In [402]:
decoder = Decoder(vocab_tar_size, embedding_dim, units, BATCH_SIZE)

sample_decoder_output, _ = decoder(tf.random.uniform((BATCH_SIZE, 1)), sample_hidden)

print(
    "Decoder output shape: (batch_size, vocab size) {}".format(
        sample_decoder_output.shape
    )
)

Decoder output shape: (batch_size, vocab size) (64, 22223)


# **Training the model**

The model is trained on a GPU machine with fixed number of epochs.

A custom training loop (instead of Model.Fit etc.) is used for which further reference is available from Tensorflow [here](https://www.tensorflow.org/guide/keras/writing_a_training_loop_from_scratch)

The model can be extended with the use of the validation data for early stopping and further fine tuning.

Checkpoints are stored for easy retrival of the model and resue without training,

In [403]:
optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction="none"
)  # Loss function is categorical crossentropy

In [404]:
def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)

    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask

    return tf.reduce_mean(loss_)

In [405]:
checkpoint_dir = "./tutorial_checkpoint_nmt"
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(optimizer=optimizer, encoder=encoder, decoder=decoder)

In [417]:
@tf.function
def train_step(inp, targ, enc_hidden):
    loss = 0

    with tf.GradientTape() as tape:
        enc_output, enc_hidden = encoder(inp, enc_hidden)

        dec_hidden = enc_hidden
        if "sos" in targ_lang.word_index:
            dec_input = tf.expand_dims([targ_lang.word_index["sos"]] * BATCH_SIZE, 1)
        else:
            dec_input = tf.equal([[targ_lang.word_index.keys()][0]] * BATCH_SIZE, 1)
        # Teacher forcing - feeding the target as the next input
        for t in range(1, targ.shape[1]):
            # passing enc_output to the decoder
            predictions, dec_hidden = decoder(dec_input, dec_hidden)

            loss += loss_function(targ[:, t], predictions)

            # using teacher forcing
            dec_input = tf.expand_dims(targ[:, t], 1)

    batch_loss = loss / int(targ.shape[1])

    variables = encoder.trainable_variables + decoder.trainable_variables

    gradients = tape.gradient(loss, variables)

    optimizer.apply_gradients(zip(gradients, variables))  # doing gradient descent

    return batch_loss

In [ ]:
train = True
EPOCHS = 2
if train:
    for epoch in range(EPOCHS):
        start = time.time()

        enc_hidden = encoder.initialize_hidden_state()
        total_loss = 0

        for batch, (inp, targ) in enumerate(dataset.take(steps_per_epoch)):
            batch_loss = train_step(inp, targ, enc_hidden)
            total_loss += batch_loss

            if batch % 100 == 0:
                print(
                    "Epoch {} Batch {} Loss {:.4f}".format(
                        epoch + 1, batch, batch_loss.numpy()
                    )
                )
        # saving (checkpoint) the model every 2 epochs
        if (epoch + 1) % 2 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)

        print("Epoch {} Loss {:.4f}".format(epoch + 1, total_loss / steps_per_epoch))
        print("Time taken for 1 epoch {} sec\n".format(time.time() - start))

In [414]:
list(targ_lang.word_index.keys())[0]

'sos'

In [420]:
checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

# **Prediction using Greedy Search**

Greedy search is used to for Decoding of text.

In [421]:
def evaluate(sentence):
    sentence = preprocess_sentence(sentence)

    inputs = [inp_lang.word_index[i] for i in sentence.split(" ")]
    inputs = tf.keras.preprocessing.sequence.pad_sequences(
        [inputs], maxlen=max_length_inp, padding="post"
    )
    inputs = tf.convert_to_tensor(inputs)

    result = ""

    hidden = [tf.zeros((1, units))]
    enc_out, enc_hidden = encoder(inputs, hidden)

    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([targ_lang.word_index["<sos>"]], 0)

    for t in range(max_length_targ):
        predictions, dec_hidden = decoder(dec_input, dec_hidden)

        # pass the encoder output, decoder hidden state(which is initialised to encoder hidden state for the first time and decoder input to the decoder)
        # make a prediction and obtain decoder hidden states

        predicted_id = tf.argmax(predictions[0]).numpy()

        result += targ_lang.index_word[predicted_id] + " "

        if targ_lang.index_word[predicted_id] == "<eos>":
            return result, sentence

        # the predicted ID is fed back into the model
        dec_input = tf.expand_dims([predicted_id], 0)

    return result, sentence

In [422]:
def translate(sentence):
    result, sentence = evaluate(sentence)

    print("Input: %s" % (sentence))
    print("Predicted translation: {}".format(result))

    return result

In [ ]:
translate("i am hungry")

# **Calculating BLEU score for evaluation**

BLEU score (Bilingual Evaluation Understudy) is calculated on the test data for evaluating the quality of translations

In [428]:
test_input_sentences = []
test_output_sentences = []

for line in open(
    os.path.join(translation_dir, "bilingual/hi-en/test.en"), encoding="utf-8"
):
    test_input_sentence = line.rstrip().strip("\n").strip("-")
    test_input_sentences.append(test_input_sentence)


for line in open(os.path.join(translation_dir, "bilingual/hi-en/test.hi")):
    test_output_sentence = line.rstrip().strip("\n").strip("-")
    line = trivial_tokenize(test_output_sentence)

    test_output_sentences.append(["<sos>"] + line + ["<eos>"])

In [429]:
print(type(test_input_sentences[90]))
print(len(test_output_sentences))
print(test_input_sentences[90])
print(test_output_sentences[90])

<class 'str'>
1000
You're slower than molasses in January.
['<sos>', 'आप', 'जनवरी', 'में', 'गुड़', 'की', 'तुलना', 'में', 'धीमी', 'है', '.', '<eos>']


In [ ]:
from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.bleu_score import SmoothingFunction

chencherry = SmoothingFunction()
evaluate_n_sentences = 10

references = []
candidates = []
for i in range(evaluate_n_sentences):
    try:
        res = translate(test_input_sentences[i])
        ref = test_output_sentences[i].copy()
        ref = [e for e in ref if e not in ("<eos>", "<sos>", ".")]
        references.append(ref)
        listToStr = " ".join(map(str, test_output_sentences[i]))
        print("Reference Translation: %s" % (listToStr))
        candidate = trivial_tokenize(res)
        candidate = [e for e in candidate if e not in ("<", "eos", ">", ".")]
        candidates.append(candidate)
    except:
        print("Sentence :", i + 1, " not translatable ..moving to next")

In [ ]:
score1 = corpus_bleu(references, candidates, smoothing_function=chencherry.method4)
score2 = corpus_bleu(references, candidates)
print("BLEU score on test data without smoothing function: ", score2)
print("BLEU score on test data with smoothing function: ", score1)

# *References*

1. The dataset used is available from [here](http://lotus.kuee.kyoto-u.ac.jp/WAT/indic-multilingual/)

2. Refer the tensorflow tutorials available on NMT [here](https://tensorflow.org/tutorials/text/nmt_with_attention) and [here](https://www.tensorflow.org/addons/tutorials/networks_seq2seq_nmt) for examples on which this notebook is modelled.

3. Refer reference code and documentation available [here](https://github.com/prashanthi-r/Eng-Hin-Neural-Machine-Translation) which has been adopted

4. Indic Library documentation can be found [here](https://github.com/anoopkunchukuttan/indic_nlp_library/blob/master/docs/indicnlp.pdf)






